# BTC-ML Strategy Research Notebook

## Objectif
Ameliorer le Sharpe ratio de 0.166 vers >0.5 en analysant:
1. L'importance des features du modele ML
2. Les seuils optimaux de confiance et volatilite
3. Les parametres de risk management (stop-loss, take-profit)
4. La frequence de retraining optimale

## Performance actuelle
- Sharpe: 0.166
- CAGR: ~5%
- Max DD: ~15%

> **[REFERENCE QC Cloud]**
> Ce notebook utilise QuantBook et necessite l'environnement QuantConnect Cloud.
> Pour executer : https://www.quantconnect.com/research


In [1]:
# Imports et setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

qb = QuantBook()
symbol = qb.add_crypto("BTCUSDT", Resolution.DAILY, Market.BINANCE).symbol
print(f"Symbol: {symbol}")

Symbol: BTCUSDT


Chargement des données historiques de BTCUSD sur 2019–2025 en résolution Daily via QuantBook pour la stratégie BTC-ML.

In [2]:
# Chargement donnees etendues (2019-2025)
start_date = datetime(2019, 1, 1)
end_date = datetime(2025, 12, 31)

history = qb.history([symbol], start_date, end_date, Resolution.DAILY)
print(f"Data loaded: {len(history)} days")
history.head()

Data loaded: 0 days


""


Détection des régimes de marché (bull/bear/sideways) sur BTCUSD via la volatilité glissante 20 jours et le momentum 126 jours.

In [3]:
# Detection des regimes de marche
df = history.droplevel(0).copy()
df['returns'] = df['close'].pct_change()
df['volatility_20'] = df['returns'].rolling(20).std() * np.sqrt(252)
df['sma_200'] = df['close'].rolling(200).mean()

# Classification des regimes
df['regime'] = 'SIDEWAYS'
df.loc[df['close'] > df['sma_200'] * 1.05, 'regime'] = 'BULL'
df.loc[df['close'] < df['sma_200'] * 0.95, 'regime'] = 'BEAR'

# Visualisation
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Prix et SMA200
axes[0].plot(df.index, df['close'], label='BTC', alpha=0.8)
axes[0].plot(df.index, df['sma_200'], label='SMA200', linestyle='--')
axes[0].set_title('BTC Price & SMA200')
axes[0].legend()
axes[0].set_yscale('log')

# Volatilite
axes[1].plot(df.index, df['volatility_20'], label='Volatility 20d', color='orange')
axes[1].axhline(y=0.60, color='r', linestyle='--', label='60% threshold')
axes[1].set_title('Annualized Volatility')
axes[1].legend()

# Regimes
regime_colors = {'BULL': 'green', 'BEAR': 'red', 'SIDEWAYS': 'gray'}
for regime in ['BULL', 'BEAR', 'SIDEWAYS']:
    mask = df['regime'] == regime
    axes[2].scatter(df.index[mask], df['close'][mask], c=regime_colors[regime], label=regime, s=10, alpha=0.5)
axes[2].set_title('Market Regimes')
axes[2].legend()
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

# Stats par regime
print("\nRegime Statistics:")
print(df.groupby('regime')['returns'].agg(['count', 'mean', 'std', 'sum']))

ValueError: Cannot remove 1 levels from an index with 1 levels: at least one level must be left.

Définition de compute_features() calculant les rendements sur 1/5/20 jours, SMA, EMA, RSI, ATR et le régime de marché, en mode strictement walk-forward pour BTC-ML.

In [4]:
# Feature engineering
def compute_features(df):
    """Compute ML features - STRICTEMENT walk-forward"""
    features = pd.DataFrame(index=df.index)
    
    # Price-based features
    features['returns_1d'] = df['close'].pct_change()
    features['returns_5d'] = df['close'].pct_change(5)
    features['returns_20d'] = df['close'].pct_change(20)
    
    # Moving averages
    features['sma_20'] = df['close'].rolling(20).mean() / df['close'] - 1
    features['ema_10'] = df['close'].ewm(span=10).mean() / df['close'] - 1
    features['ema_50'] = df['close'].ewm(span=50).mean() / df['close'] - 1
    features['ema_200'] = df['close'].ewm(span=200).mean() / df['close'] - 1
    
    # RSI
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss.replace(0, 1e-10)
    features['rsi_14'] = 100 - (100 / (1 + rs))
    
    # Volatility
    features['volatility_20'] = df['returns'].rolling(20).std() * np.sqrt(252)
    features['atr_14'] = (df['high'] - df['low']).rolling(14).mean() / df['close']
    
    # Momentum
    features['momentum_10'] = df['close'] / df['close'].shift(10) - 1
    features['momentum_20'] = df['close'] / df['close'].shift(20) - 1
    
    # Trend strength
    features['price_vs_sma200'] = df['close'] / df['close'].rolling(200).mean() - 1
    
    return features

features = compute_features(df)
features['target'] = (df['close'].shift(-1) > df['close']).astype(int)  # Next day direction

# Nettoyage
features = features.dropna()
print(f"Features shape: {features.shape}")
features.head()

NameError: name 'df' is not defined

Préparation des données d'entraînement et de test pour le modèle de machine learning.

In [5]:
# Analyse de l'importance des features
train_end = datetime(2022, 12, 31)
train_data = features[features.index <= train_end]

feature_cols = [c for c in features.columns if c != 'target']
X_train = train_data[feature_cols].values
y_train = train_data['target'].values

# Training
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(importance.tail(5))

NameError: name 'features' is not defined

Test des seuils ZigZag (0.40, 0.50, 0.60, 0.70) pour comparer la qualité des canaux détectés sur les données BTC.

In [6]:
# Grid search: Volatility threshold optimal
test_data = features[features.index > train_end]
X_test = test_data[feature_cols].values
y_test = test_data['target'].values

# Probabilites
proba = rf.predict_proba(X_test)
confidence = proba[:, 1]  # Probabilite de hausse

# Test differents seuils de volatilite
vol_thresholds = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.0]
results_vol = []

for vol_thresh in vol_thresholds:
    mask = test_data['volatility_20'].values <= vol_thresh
    if mask.sum() < 50:
        continue
    
    y_test_masked = y_test[mask]
    pred_masked = (confidence[mask] > 0.5).astype(int)
    
    acc = accuracy_score(y_test_masked, pred_masked)
    
    # Simuler returns avec filtre vol
    returns = test_data['returns_1d'].values[mask]
    positions = np.where(confidence[mask] > 0.55, 1, 0)
    strategy_returns = positions * returns
    
    sharpe = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252) if np.std(strategy_returns) > 0 else 0
    win_rate = (strategy_returns > 0).sum() / len(strategy_returns) if len(strategy_returns) > 0 else 0
    
    results_vol.append({
        'vol_threshold': vol_thresh,
        'accuracy': acc,
        'sharpe': sharpe,
        'win_rate': win_rate,
        'trades': mask.sum()
    })

results_vol_df = pd.DataFrame(results_vol)
print("\nVolatility Threshold Analysis:")
print(results_vol_df)

best_vol = results_vol_df.loc[results_vol_df['sharpe'].idxmax()]
print(f"\nBest volatility threshold: {best_vol['vol_threshold']:.0%} (Sharpe: {best_vol['sharpe']:.3f})")

NameError: name 'features' is not defined

Recherche du seuil de confiance optimal pour les signaux ML (précision/rappel) dans la stratégie BTC-ML.

In [7]:
# Grid search: Confidence threshold optimal
confidence_thresholds = [(0.45, 0.55), (0.50, 0.50), (0.55, 0.45), (0.60, 0.40), (0.65, 0.35)]
results_conf = []

for long_thresh, exit_thresh in confidence_thresholds:
    # Positions basees sur confidence
    positions = np.zeros(len(test_data))
    for i in range(len(test_data)):
        if confidence[i] > long_thresh:
            positions[i] = 1
        elif confidence[i] < exit_thresh:
            positions[i] = 0
    
    returns = test_data['returns_1d'].values
    strategy_returns = positions * returns
    
    sharpe = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252) if np.std(strategy_returns) > 0 else 0
    total_return = (1 + strategy_returns).prod() - 1
    max_dd = (strategy_returns.cumsum() - np.maximum.accumulate(strategy_returns.cumsum())).min()
    
    results_conf.append({
        'long_threshold': long_thresh,
        'exit_threshold': exit_thresh,
        'sharpe': sharpe,
        'total_return': total_return,
        'max_drawdown': max_dd,
        'trades': (np.diff(positions) != 0).sum()
    })

results_conf_df = pd.DataFrame(results_conf)
print("\nConfidence Threshold Analysis:")
print(results_conf_df)

best_conf = results_conf_df.loc[results_conf_df['sharpe'].idxmax()]
print(f"\nBest confidence: Long > {best_conf['long_threshold']:.2f}, Exit < {best_conf['exit_threshold']:.2f}")
print(f"Expected Sharpe: {best_conf['sharpe']:.3f}")

NameError: name 'test_data' is not defined

Validation walk-forward en 5 fenêtres glissantes (TimeSeriesSplit) pour mesurer la robustesse hors-échantillon de BTC-ML.

In [8]:
# Walk-forward validation
print("\n=== WALK-FORWARD VALIDATION ===")

# Split en 5 periodes
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

wf_results = []
train_score_total = 0
test_score_total = 0

for fold, (train_idx, test_idx) in enumerate(tscv.split(features)):
    X_train_wf = features.iloc[train_idx][feature_cols].values
    y_train_wf = features.iloc[train_idx]['target'].values
    X_test_wf = features.iloc[test_idx][feature_cols].values
    y_test_wf = features.iloc[test_idx]['target'].values
    
    rf_wf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_wf.fit(X_train_wf, y_train_wf)
    
    train_acc = rf_wf.score(X_train_wf, y_train_wf)
    test_acc = rf_wf.score(X_test_wf, y_test_wf)
    
    train_score_total += train_acc
    test_score_total += test_acc
    
    wf_results.append({
        'fold': fold + 1,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
        'train_acc': train_acc,
        'test_acc': test_acc
    })
    print(f"Fold {fold+1}: Train acc={train_acc:.2%}, Test acc={test_acc:.2%}")

wf_df = pd.DataFrame(wf_results)
avg_train = train_score_total / n_splits
avg_test = test_score_total / n_splits
ratio = avg_test / avg_train if avg_train > 0 else 0

print(f"\nAverage Train: {avg_train:.2%}")
print(f"Average Test: {avg_test:.2%}")
print(f"OOS/IS Ratio: {ratio:.1%}")

if ratio > 0.6:
    print("PASS: Model generalizes well (ratio > 60%)")
else:
    print("FAIL: Model overfits (ratio < 60%)")


=== WALK-FORWARD VALIDATION ===


NameError: name 'features' is not defined

## Synthese et Recommandations

### Parametres optimaux identifies

D'apres les analyses ci-dessus, voici les parametres recommandes:

In [9]:
# Resume des recommandations
print("\n" + "="*60)
print("RECOMMANDATIONS POUR BTC-ML")
print("="*60)

recommendations = {
    'volatility_threshold': best_vol['vol_threshold'],
    'confidence_long': best_conf['long_threshold'],
    'confidence_exit': best_conf['exit_threshold'],
    'expected_sharpe': max(best_vol['sharpe'], best_conf['sharpe']),
    'walk_forward_ratio': ratio
}

print(f"""
Parametres actuels:
- VOLATILITY_THRESHOLD = 0.60
- CONFIDENCE_LONG_THRESHOLD = 0.55
- CONFIDENCE_EXIT_THRESHOLD = 0.45
- STOP_LOSS_PCT = 0.10
- TAKE_PROFIT_PCT = 0.20

Parametres recommandes:
- VOLATILITY_THRESHOLD = {recommendations['volatility_threshold']:.0%}
- CONFIDENCE_LONG_THRESHOLD = {recommendations['confidence_long']:.2f}
- CONFIDENCE_EXIT_THRESHOLD = {recommendations['confidence_exit']:.2f}

Sharpe attendu: {recommendations['expected_sharpe']:.3f}
Walk-forward ratio: {recommendations['walk_forward_ratio']:.1%}
""")

# Export JSON pour application
import json
print("\nJSON pour mise a jour du code:")
print(json.dumps(recommendations, indent=2))


RECOMMANDATIONS POUR BTC-ML


NameError: name 'best_vol' is not defined